# hatchvision — training, Hebbian memory & explainability demo

This notebook walks the full pipeline end to end:

1. **Load a dataset** through the pluggable loader interface (swap datasets by changing *one line*).
2. **Build a classifier** with any registered backbone — including the experimental **Baby Dragon Hatchling (BDH)** encoder.
3. **Attach the Hebbian feature memory**, which records neuron co-activation during training. It is *observation only*: hooks detach everything, so optimization is bit-identical with or without it (this is covered by a unit test).
4. Evaluate, then compute **feature importance** with **Grad-CAM** and **SHAP**.
5. **Cluster** the Hebbian co-activation matrix into **visual concepts**, map them to classes, and inspect exemplar images.
6. **Export the concept graph as IVGraph JSON** and explore it in the bundled Vercel web app (`webapp/`).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root

import torch
import matplotlib.pyplot as plt

from hatchvision import (
    HebbianFeatureMemory, TrainConfig, Trainer,
    build_loader, create_model,
)
from hatchvision.data import available_loaders
from hatchvision.models import available_backbones

print("datasets: ", available_loaders())
print("backbones:", available_backbones())

## 1. Dataset

Everything downstream is driven by the loader's `DatasetSpec` (class names, image size, channels, normalization). **To use a different dataset, change only this cell** — e.g. `build_loader("fashion_mnist")`, or point `build_loader("imagefolder", root="/path/to/data")` at any `train/val` directory of class folders.

We cap the split sizes so the demo trains in minutes on CPU; remove the limits for a real run.

In [ ]:
DATASET = "cifar10"          # <- the ONLY line to change for a new dataset
data = build_loader(DATASET, root="../data", limit_train=6000, limit_val=1500)
spec = data.spec
train_loader, val_loader = data.dataloaders(batch_size=64)
print(spec)

## 2. Model — pick a backbone

Backbones implement one small interface (`forward`, `feature_dim`, `cam_layer`, `hebbian_layers`), so they are fully interchangeable. Options include `simple_cnn` (fast, good for demos), `resnet18/34/50`, and `bdh` — an **experimental vision adaptation of the Baby Dragon Hatchling architecture** (arXiv:2509.26507): patches are lifted into a high-dimensional *sparse, positive* neuron space with linear attention, which makes its activations especially well-suited to Hebbian co-activation analysis.

In [ ]:
BACKBONE = "simple_cnn"      # try "bdh" or "resnet18"
model = create_model(BACKBONE, spec)
n_params = sum(p.numel() for p in model.parameters())
print(f"{BACKBONE}: {n_params/1e6:.2f} M params")
print("hebbian layers:", list(model.hebbian_layers()))

## 3. Attach the Hebbian feature memory & train

The memory hooks the backbone's `hebbian_layers()` and keeps an EMA of the outer product of (pooled, rectified, L2-normalized) activations — *"neurons that fire together, wire together."* The trainer forwards batch labels to it so class-conditional firing rates accumulate too. It never touches gradients or activations, so it changes nothing about optimization.

In [ ]:
memory = HebbianFeatureMemory(model, num_classes=spec.num_classes, max_units=128)
trainer = Trainer(model, TrainConfig(epochs=3, lr=3e-4, log_every=25), hebbian_memory=memory)
history = trainer.fit(train_loader, val_loader)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3), facecolor="#fcfcfb")
for ax, metric in zip(axes, ["loss", "acc"]):
    ax.plot(history[f"train_{metric}"], color="#2a78d6", lw=2, label="train")
    if history[f"val_{metric}"]:
        ax.plot(history[f"val_{metric}"], color="#1baf7a", lw=2, label="val")
    ax.set_xlabel("epoch"); ax.set_title(metric, color="#0b0b0b")
    ax.legend(frameon=False)
    ax.grid(color="#e1e0d9", lw=0.6); ax.set_axisbelow(True)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## 4a. Feature importance — Grad-CAM

Grad-CAM asks *where in the image* the evidence for a prediction is: it weights the backbone's last spatial feature maps by their pooled gradients. Every backbone advertises its own `cam_layer()`, so this cell is architecture-agnostic.

In [ ]:
from hatchvision.explain import GradCAM, denormalize

images, labels = next(iter(val_loader))
images, labels = images[:8], labels[:8]

with GradCAM(model) as cam:
    heatmaps = cam(images.clone().to(trainer.device))

display = denormalize(images, spec.mean, spec.std).permute(0, 2, 3, 1).cpu()
with torch.no_grad():
    preds = model(images.to(trainer.device)).argmax(1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4), facecolor="#fcfcfb")
for i in range(8):
    img = display[i] if display.shape[-1] == 3 else display[i, ..., 0]
    axes[0, i].imshow(img, cmap=None if display.shape[-1] == 3 else "gray")
    axes[0, i].set_title(
        f"{spec.class_names[preds[i]]}\n(true {spec.class_names[labels[i]]})",
        fontsize=8, color="#0b0b0b")
    axes[1, i].imshow(img, cmap=None if display.shape[-1] == 3 else "gray")
    axes[1, i].imshow(heatmaps[i].cpu(), cmap="magma", alpha=0.55)
    for r in range(2): axes[r, i].axis("off")
axes[1, 0].set_ylabel("Grad-CAM")
plt.suptitle("Grad-CAM: where the model looks", color="#0b0b0b")
plt.tight_layout(); plt.show()

## 4b. Feature importance — SHAP

SHAP (expected gradients) attributes the prediction to individual pixels against a background distribution: red pushes *toward* the predicted class, blue *away*. It is an optional dependency (`pip install shap`), so this cell degrades gracefully.

In [ ]:
from hatchvision.explain import shap_available

if shap_available():
    from hatchvision.explain import ShapExplainer
    background = next(iter(train_loader))[0][:48]
    explainer = ShapExplainer(model, background)
    sv = explainer.explain(images[:4], nsamples=50)   # [4, H, W]

    fig, axes = plt.subplots(1, 4, figsize=(10, 3), facecolor="#fcfcfb")
    bound = abs(sv).max()
    for i, ax in enumerate(axes):
        img = display[i] if display.shape[-1] == 3 else display[i, ..., 0]
        ax.imshow(img, cmap=None if display.shape[-1] == 3 else "gray")
        ax.imshow(sv[i], cmap="coolwarm", vmin=-bound, vmax=bound, alpha=0.6)
        ax.set_title(spec.class_names[preds[i]], fontsize=9, color="#0b0b0b")
        ax.axis("off")
    plt.suptitle("SHAP pixel attribution (red = supports prediction)", color="#0b0b0b")
    plt.tight_layout(); plt.show()
else:
    print("shap not installed — run `pip install shap` to enable this section")

## 5. From Hebbian co-activation to visual concepts

The memory now holds, for each observed layer, a co-activation matrix over hidden units. Units that consistently fire together are clustered (agglomerative, distance = 1 − correlation); each cluster is a candidate **concept**. Class-conditional firing rates map every concept to the labels it responds to, and a probe pass finds the images that excite it most — its **exemplars**.

In [ ]:
from hatchvision.explain import cluster_concepts, find_exemplars

LAYER = memory.layer_names[-1]      # deepest observed layer
corr = memory.correlation(LAYER)

fig, ax = plt.subplots(figsize=(5, 4), facecolor="#fcfcfb")
im = ax.imshow(corr, cmap="Blues", vmin=0, vmax=1)
ax.set_title(f"Hebbian co-activation — {LAYER}", color="#0b0b0b")
ax.set_xlabel("unit"); ax.set_ylabel("unit")
plt.colorbar(im, label="co-activation")
plt.tight_layout(); plt.show()

concepts = cluster_concepts(memory, LAYER, spec.class_names, n_concepts=8)
for c in concepts:
    aff = ", ".join(f"{k} {v:.2f}" for k, v in c.class_affinity.items())
    print(f"concept {c.concept_id:2d}  [{c.label:24s}] "
          f"{len(c.units):3d} units  coherence {c.coherence:.2f}  ({aff})")

In [ ]:
# Exemplars: probe images that activate each concept most strongly
probe_images = torch.stack([val_loader.dataset[i][0] for i in range(256)])
find_exemplars(concepts, memory, model, probe_images, top_k=6)

show = concepts[:4]
fig, axes = plt.subplots(len(show), 6, figsize=(9, 1.7 * len(show)), facecolor="#fcfcfb")
probe_disp = denormalize(probe_images, spec.mean, spec.std).permute(0, 2, 3, 1)
for r, concept in enumerate(show):
    for k, idx in enumerate(concept.exemplars):
        img = probe_disp[idx] if probe_disp.shape[-1] == 3 else probe_disp[idx, ..., 0]
        axes[r, k].imshow(img, cmap=None if probe_disp.shape[-1] == 3 else "gray")
        axes[r, k].axis("off")
    axes[r, 0].set_ylabel(concept.label, rotation=0, ha="right", fontsize=9)
plt.suptitle("Concept exemplars (rows = Hebbian clusters)", color="#0b0b0b")
plt.tight_layout(); plt.show()

## 6. Export to IVGraph and visualize

The concept graph — units, concepts, classes, and their co-activation / membership / affinity edges — serializes to **IVGraph JSON**. The bundled static web app renders it; to publish, `cd webapp && npx vercel deploy`.

In [ ]:
from hatchvision.export import export_ivgraph

path = export_ivgraph(
    memory, concepts, LAYER, spec.class_names,
    "../exports/graph.json",
    meta={"dataset": spec.name, "backbone": BACKBONE},
)
print("wrote", path)

# Make it the web app's default graph:
import shutil
shutil.copy(path, "../webapp/sample-graph.json")
print("webapp updated — preview with:  cd webapp && python3 -m http.server 8000")

### Where to go from here

- **Swap the dataset**: change the `build_loader(...)` line in step 1 — nothing else. For a custom dataset, use the `imagefolder` loader or register your own `DatasetLoader` subclass.
- **Swap the encoder**: change `BACKBONE`, or register a new `Backbone` with `@register_backbone("name")`.
- **BDH**: rerun with `BACKBONE = "bdh"` — its sparse positive neuron space produces markedly crisper Hebbian clusters.
- **CLI**: the same pipeline runs headless via `python scripts/train.py --dataset cifar10 --backbone bdh --hebbian --export-graph exports/graph.json`.